# Entrenamiento de los modelos YOLO

Entrena los dos detectores del proyecto con `src/train/train.py` y la configuración común `configs/train_base.yaml`:

| Modelo | Datos | Entregables |
|---|---|---|
| `full_yolov8s` | CCT completo (`data_full.yaml`) | E1, E2, E4, E5 |
| `eccv_yolov8s` | Partición oficial ECCV (`data_eccv.yaml`) | E3 |

Las corridas se guardan en Drive (`MyDrive/redes-parcial-p3/runs/`) para no perderlas si se cierra la sesión y poder reanudarlas.

Usar un entorno con GPU **L4** o **A100** (más disco y más rápido que la T4).

## Preparar la máquina

Cada notebook de Colab corre en su propia máquina, así que primero se ejecuta `colab_setup.ipynb` completo: monta Drive, clona el repo, instala dependencias, extrae las imágenes y genera las etiquetas YOLO. La extracción de los 105 GB tarda bastante.

In [1]:
import os
from google.colab import drive

drive.mount('/content/drive')
REPO_DIR = '/content/redes-parcial-p3'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/Roxana-Vargas/redes-parcial-p3.git {REPO_DIR}
%cd {REPO_DIR}
!git pull
%run notebooks/colab_setup.ipynb

Ultralytics 8.4.187 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
Setup complete ✅ (12 CPUs, 53.0 GB RAM, 42.6/235.7 GB disk)
cct_images.tar.gz ya está completo en Drive.
Disco libre en /content: 207 GB
Extrayendo desde LILA...
1
2
3
4
6
7
8
9
11
12
13
15
16
17
18
20
21
22
23
25
26
27
28
30
31
32
33
35
36
37
38
39
41
42
43
44
46
47
48
49
51
52
53
54
56
57
58
59
61
62
63
64
66
67
68
69
71
72
73
74
76
77
78
79
81
82
83
84
86
87
88
89
91
92
93
94
96
97
98
99
100
189M	/content/redes-parcial-p3/data/raw/annotations
106G	/content/redes-parcial-p3/data/raw/cct_images
Imágenes .jpg: 244,584
Anotaciones JSON:
   annotations/CaltechCameraTrapsSplits_v0.json
   annotations/caltech_bboxes_20200316.json
   annotations/caltech_images_20210113.json
   annotations/eccv_18_annotation_files/cis_test_annotations.json
   annotations/eccv_18_annotation_files/cis_val_annotations.json
   annotations/eccv_18_annotation_files/train_annotations.json
   annotations/eccv_18_annotation_files/tra

In [2]:
PROJECT = ' /content/drive/MyDrive/redes-parcial-p3/runs'
!cat configs/train_base.yaml

# Hiperparámetros comunes al modelo principal y al modelo E3.
# Justificación de cada valor en docs/decisiones_entrenamiento.md

imgsz: 640
epochs: 40
patience: 10          # early stopping si el mAP de validación no mejora en 10 épocas
batch: -1             # autobatch: el mayor batch que cabe en ~60 % de la VRAM de la GPU
optimizer: auto
seed: 0
deterministic: true
workers: 8
close_mosaic: 10      # desactiva mosaic en las últimas 10 épocas
plots: true

# Aumento de datos
hsv_h: 0.015
hsv_s: 0.7
hsv_v: 0.4
degrees: 0.0
translate: 0.1
scale: 0.5
shear: 0.0
perspective: 0.0
fliplr: 0.5
flipud: 0.0
mosaic: 1.0
mixup: 0.0


## Corrida de prueba

Una época con el 2 % del train, para verificar que los datos, las etiquetas y la GPU funcionan antes de lanzar el entrenamiento largo. Si falla aquí, no se pierden horas de GPU.

In [3]:
!python src/train/train.py --data data/yolo/data_full.yaml --name prueba --project {PROJECT} --epochs 1 --fraction 0.02

Se truncaron las últimas líneas 5000 del resultado de transmisión.
val: /content/redes-parcial-p3/data/yolo/images/59dc2813-23d2-11e8-a6a3-ec086b02610b.jpg: corrupt JPEG restored and saved
val: /content/redes-parcial-p3/data/yolo/images/59dc2816-23d2-11e8-a6a3-ec086b02610b.jpg: corrupt JPEG restored and saved
val: /content/redes-parcial-p3/data/yolo/images/59dc2817-23d2-11e8-a6a3-ec086b02610b.jpg: corrupt JPEG restored and saved
val: /content/redes-parcial-p3/data/yolo/images/59ddbf8a-23d2-11e8-a6a3-ec086b02610b.jpg: corrupt JPEG restored and saved
val: /content/redes-parcial-p3/data/yolo/images/59ddbf8b-23d2-11e8-a6a3-ec086b02610b.jpg: corrupt JPEG restored and saved
val: /content/redes-parcial-p3/data/yolo/images/59ddbf8f-23d2-11e8-a6a3-ec086b02610b.jpg: corrupt JPEG restored and saved
val: /content/redes-parcial-p3/data/yolo/images/59ddbfa2-23d2-11e8-a6a3-ec086b02610b.jpg: corrupt JPEG restored and saved
val: /content/redes-parcial-p3/data/yolo/images/59ddbfaa-23d2-11e8-a6a3-ec086b0

In [10]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Modelo principal (CCT completo)

Entrenamiento largo: unas 113k imágenes por época (83 % vacías). Revisar de vez en cuando que la sesión siga activa.

In [11]:
!python src/train/train.py --data data/yolo/data_full.yaml --name full_yolov8s --project {PROJECT}

New https://pypi.org/project/ultralytics/8.4.188 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.187 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=configs/train_base.yaml, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/yolo/data_full.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8

## Modelo E3 (partición oficial ECCV)

Misma configuración, entrenado solo con `eccv_train` (unas 11k imágenes). Se valida en `eccv_cis_val`.

In [4]:
!python src/train/train.py --data data/yolo/data_eccv.yaml --name eccv_yolov8s --project {PROJECT}

New https://pypi.org/project/ultralytics/8.4.188 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.187 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=configs/train_base.yaml, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/yolo/data_eccv.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8

## Reanudar una corrida cortada

Si la sesión de Colab se desconecta a mitad del entrenamiento, volver a preparar la máquina (primera celda) y reanudar desde el último checkpoint guardado en Drive. Cambiar `RUN` por la corrida que se cortó.

In [ ]:
RUN = 'full_yolov8s'
!python src/train/train.py --name {RUN} --project {PROJECT} --resume

## Curvas de entrenamiento

Pérdidas y métricas por época que guarda Ultralytics en cada corrida.

In [ ]:
from IPython.display import Image, display

for run in ['full_yolov8s', 'eccv_yolov8s']:
    path = os.path.join(PROJECT, run, 'results.png')
    if os.path.exists(path):
        print(run)
        display(Image(filename=path, width=900))

In [7]:
!ls /content/redes-parcial-p3/data/raw/cct_images | wc -l

161168
